In [1]:
import kagglehub
import shutil
from pathlib import Path

path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci")
print("Downloaded to:", path)

raw_dir = Path("../data/raw/online_retail_ii")
raw_dir.mkdir(parents=True, exist_ok=True)

for item in Path(path).iterdir():
    target = raw_dir / item.name
    if item.is_file():
        shutil.copy2(item, target)
        print(f"Copied: {item.name}")

print("✅ Dataset ready!")

100%|█████████████████████████████████████████████████████████████████████████████| 14.5M/14.5M [00:00<00:00, 42.7MB/s]

Extracting files...


Downloaded to: C:\Users\Tyush01\.cache\kagglehub\datasets\mashlyn\online-retail-ii-uci\versions\3
Copied: online_retail_II.csv
✅ Dataset ready!


In [2]:
from pathlib import Path

files = list(Path("../data/raw/online_retail_ii").glob("*"))
files

[WindowsPath('../data/raw/online_retail_ii/online_retail_II.csv')]

In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/online_retail_ii/online_retail_II.csv", encoding="latin1")
df.head() 
# Preview the first few rows


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [14]:
df.shape, df.columns


((1067371, 8),
 Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
        'Price', 'CustomerID', 'Country'],
       dtype='object'))

In [15]:
# Review column data types, non-null counts, and memory usage
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   CustomerID   824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [16]:
# Count missing values in each column
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
CustomerID     243007
Country             0
dtype: int64

In [17]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [18]:
# Standardize column names and convert InvoiceDate to datetime
df = df.rename(columns={"Customer ID": "CustomerID"})
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

In [19]:
# Create a revenue column
df["Revenue"] = df["Quantity"] * df["Price"]

In [20]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   CustomerID   824364 non-null   float64       
 7   Country      1067371 non-null  object        
 8   Revenue      1067371 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 73.3+ MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [21]:
# Remove rows with missing customer IDs and negative/zero quantity or price
df_clean = df.dropna(subset=["CustomerID"]).copy()
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["Price"] > 0)]

# Convert CustomerID to integer after removing missing values
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)

# Preview cleaned data
df_clean.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [23]:

df_clean.shape, df_clean.isna().sum()

((805549, 9),
 Invoice        0
 StockCode      0
 Description    0
 Quantity       0
 InvoiceDate    0
 Price          0
 CustomerID     0
 Country        0
 Revenue        0
 dtype: int64)

In [24]:

df_clean.to_csv("../data/processed/online_retail_cleaned.csv", index=False)

In [25]:
from pathlib import Path

Path("../data/processed/online_retail_cleaned.csv").exists()

True